# Nonparametric Ab Test Simulation

Схема планирования A/B-теста для непрерывной метрики,
когда классический t-test применять неудобно или рискованно.

В файле собраны:
- шаблоны для бинарных и непрерывных метрик;
- основной рабочий сценарий для ненормального распределения;
- подбор размера выборки через Monte Carlo.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import laplace_asymmetric, mannwhitneyu, ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.stats.api as sms

## 1. Шаблоны для статистических тестов

Ниже оставлены "шпаргалки" по самым частым сценариям.
Они нужны как быстрый ориентир:
что считать для effect size, какой тест запускать, какую функцию использовать.

In [ ]:
# Бинарная метрика:
# proportions_ztest сравнивает две доли успеха.
effect_size = sms.proportion_effectsize(baseline, baseline + mde)
sample_size = sms.NormalIndPower().solve_power(effect_size, power=0.80, alpha=0.05, ratio=1)
_, pvalue = proportions_ztest([group_a.sum(), group_b.sum()], [len(group_a), len(group_b)])

In [ ]:
# Непрерывная метрика при применимости t-test:
# effect_size = MDE / std исторических данных
effect_size = mde / historical_metric.std()
sample_size = sms.TTestIndPower().solve_power(effect_size, power=0.80, alpha=0.05, ratio=1)
_, pvalue = ttest_ind(group_a, group_b, equal_var=False)

In [ ]:
# Непрерывная метрика при выборе непараметрического теста:
_, pvalue = mannwhitneyu(group_a, group_b)

## 2. Шаблон: A/B-тест для бинарных метрик

In [ ]:
# Сценарий:
# 1) считаем размер выборки аналитически;
# 2) проверяем через Monte Carlo, что тест дает нужную мощность и нужный уровень ошибок.
baseline = 0.2
mde = 0.05

# В исходной работе размер выборки был заранее известен:
sample_size = 1092

# 3.1. Есть реальная разница между группами:
# хотим, чтобы p-value < 0.05 возникало примерно в 80% случаев.
pvalues_with_effect = []

for _ in range(10000):
    group_a = np.random.binomial(1, baseline, sample_size)
    group_b = np.random.binomial(1, baseline + mde, sample_size)
    _, pvalue = proportions_ztest([group_a.sum(), group_b.sum()], [len(group_a), len(group_b)])
    pvalues_with_effect.append(pvalue)

pvalues_with_effect = pd.Series(pvalues_with_effect)
print((pvalues_with_effect < 0.05).mean())

# 3.2. Разницы нет:
# хотим, чтобы ложное срабатывание было около 5%.
pvalues_without_effect = []

for _ in range(10000):
    group_a = np.random.binomial(1, baseline, sample_size)
    group_b = np.random.binomial(1, baseline, sample_size)
    _, pvalue = proportions_ztest([group_a.sum(), group_b.sum()], [len(group_a), len(group_b)])
    pvalues_without_effect.append(pvalue)

pvalues_without_effect = pd.Series(pvalues_without_effect)
print((pvalues_without_effect < 0.05).mean())

## 3. Шаблон: A/B-тест для непрерывной метрики

In [ ]:
# Здесь показан более простой пример на логнормальном распределении.
pvalues_with_effect = []

for _ in range(1000):
    group_a = np.random.lognormal(mean=5, sigma=2, size=40000)
    group_b = np.random.lognormal(mean=5.04, sigma=2, size=40000)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_with_effect.append(pvalue)

pvalues_with_effect = pd.Series(pvalues_with_effect)
print((pvalues_with_effect < 0.05).mean())

pvalues_without_effect = []

for _ in range(1000):
    group_a = np.random.lognormal(mean=5, sigma=2, size=40000)
    group_b = np.random.lognormal(mean=5, sigma=2, size=40000)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_without_effect.append(pvalue)

pvalues_without_effect = pd.Series(pvalues_without_effect)
print((pvalues_without_effect < 0.05).mean())

## 4. Идея выбора подхода для непрерывной метрики

Для непрерывной величины здесь рассматриваются два пути.

1. Если t-test применим и его предпосылки выполняются:
   - считаем размер выборки через statsmodels;
   - проводим A/B-тест;
   - применяем t-test и интерпретируем p-value.

2. Если распределение тяжелохвостое, асимметричное или t-test вызывает сомнения:
   - подбираем форму исторического распределения;
   - через Monte Carlo подбираем sample size;
   - на итоговых группах применяем тест Манна–Уитни.

## 5. Основной рабочий сценарий: планирование A/B-теста для ненормальных данных

Исторический набор данных по ценам домов.

In [ ]:
housing_df = pd.read_csv('data/ames_housing.txt', sep='\t', header=0, index_col=False)

В этом примере метрика — SalePrice.

Цель: понять, какую выборку нужно собрать,
чтобы детектировать заданный сдвиг среднего.

In [ ]:
baseline = housing_df['SalePrice'].mean()
mde = 2000

После подбора формы распределения были зафиксированы параметры:

kappa и scale — параметры распределения;

loc считается через формулу для среднего значения.

In [ ]:
sample_size = 13500
kappa = 0.56
scale = 42949.68

Формула для loc:

loc = mean - scale * (1 / kappa - kappa)

In [ ]:
loc_a = baseline - scale * (1 / kappa - kappa)
loc_b = (baseline + mde) - scale * (1 / kappa - kappa)

data_a соответствует историческому сценарию,
data_b — сценарию с улучшением метрики на MDE.

In [ ]:
data_a = laplace_asymmetric(kappa=kappa, loc=loc_a, scale=scale)
data_b = laplace_asymmetric(kappa=kappa, loc=loc_b, scale=scale)

## 6. Проверка sample size через Monte Carlo

Сценарий 1: 

различие между группами действительно есть.

In [ ]:
pvalues_with_effect = []

In [ ]:
for _ in range(1000):
    group_a = data_a.rvs(size=sample_size)
    group_b = data_b.rvs(size=sample_size)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_with_effect.append(pvalue)

In [ ]:
pvalues_with_effect = pd.Series(pvalues_with_effect)
print((pvalues_with_effect < 0.05).mean())

Сценарий 2: 

различия нет, обе группы из одного распределения.
Здесь считаем долю ложных срабатываний.

In [ ]:
pvalues_without_effect = []

In [ ]:
for _ in range(1000):
    group_a = data_a.rvs(size=sample_size)
    group_b = data_a.rvs(size=sample_size)
    _, pvalue = mannwhitneyu(group_a, group_b, alternative='two-sided')
    pvalues_without_effect.append(pvalue)

In [ ]:
pvalues_without_effect = pd.Series(pvalues_without_effect)
print((pvalues_without_effect < 0.05).mean())

## 7. Итоговая интерпретация

В исходной работе рассматривались два примера:
- sample_size = 140 при mde = 20000;
- sample_size = 13500 при mde = 2000.

Логика одна и та же:


если маленький эффект, то нужна гораздо большая выборка,
чтобы с той же мощностью 80% и alpha=5% его детектировать.

---
**Примечание:** для запуска ноутбука достаточно открыть его в корне соответствующего репозитория — все файлы данных лежат рядом и читаются по относительному пути.